# PRISM Rebuttal round 2 - proportional label budget

Reviewer tp5b:

> I also noticed that temperature scaling uses the complete validation set at
> every dataset fraction. The complete validation set is also used for early
> stopping. Therefore, the 1% condition is not really a 1% total-label
> condition.

This is correct as stated. Our protocol subsamples the training split by the
label fraction while holding the validation split at full size, so a "1%"
cell consumes 1% of train plus 100% of val. On MHIST that is 18 training
labels alongside 326 validation labels, and the label budget is dominated by
the validation set.

This notebook re-runs the full in-distribution protocol under a **proportional
budget**: the validation split is subsampled by the same fraction as the
training split, class-stratified, from the same seed. At 1% a cell then uses
1% of train and 1% of val, which is what "1% of the available labels" should
mean.

**What this can and cannot change.** Raw ECE and AUROC never touch the
validation split, so those columns are unaffected by construction and serve as
a consistency check: they should reproduce `indomain_all_v2.csv` exactly.
Temperature, scaled ECE, and anything derived from them (CRI, calibration
recoverability) can move, and the point of the run is to find out by how much.

864 logistic-regression fits, CPU only, roughly 30 to 60 minutes.
Checkpointed per (model, dataset).

In [ ]:
import os, gc, glob, time, warnings
import numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score, brier_score_loss
from scipy.optimize import minimize_scalar
from google.colab import drive

warnings.filterwarnings('ignore')
drive.mount('/content/drive')

BASE    = '/content/drive/MyDrive/PRISM'
EMB_DIR = f'{BASE}/embeddings'
OUT_DIR = f'{BASE}/results_v2'
CKPT    = f'{OUT_DIR}/propval_parts'
os.makedirs(CKPT, exist_ok=True)

MODELS = ['CLIP','PLIP','CONCH','VIRCHOW2','UNI','GigaPath','H-Optimus-0','MIDNIGHT']
MKEYS  = ['clip','plip','conch','virchow2','uni','gigapath','h_optimus_0','midnight']
M2K    = dict(zip(MODELS, MKEYS))

DATASETS = ['PCam','BRACS','CRC','MHIST','LungHist700','SPIDER-Breast']
DKEYS    = ['pcam','bracs','crc','mhist','lunghist700','spider_breast']
D2K      = dict(zip(DATASETS, DKEYS))

FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.00]
SEEDS     = [42, 123, 456]
N_BINS, C_DEFAULT, MAX_ITER = 15, 1.0, 1000

ORDER = ['LungHist700','MHIST','BRACS','CRC','SPIDER-Breast','PCam']
print('grid:', len(MODELS)*len(DATASETS)*len(FRACTIONS)*len(SEEDS), 'fits')

Mounted at /content/drive
grid: 864 fits


## 1. Helpers, identical to the corrected protocol

In [ ]:
def _ece_edges(conf, correct, edges):
    ece, n = 0.0, len(conf)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf >= lo) & (conf < hi)
        if m.sum() > 0:
            ece += m.sum() * abs(correct[m].mean() - conf[m].mean())
    return float(ece / n)

def conf_and_correct(proba, y):
    if proba.shape[1] == 2:
        return proba[:, 1], (y == 1).astype(float)
    return proba.max(1), (proba.argmax(1) == y).astype(float)

def ece_fixed(proba, y, n_bins=N_BINS):
    c, k = conf_and_correct(proba, y)
    return _ece_edges(c, k, np.linspace(0, 1, n_bins + 1))

def ece_adaptive(proba, y, n_bins=N_BINS):
    c, k = conf_and_correct(proba, y)
    e = np.quantile(c, np.linspace(0, 1, n_bins + 1))
    e[0], e[-1] = 0.0, 1.0 + 1e-9
    e = np.unique(e)
    return ece_fixed(proba, y, n_bins) if len(e) < 3 else _ece_edges(c, k, e)

def softmax(z):
    z = z - z.max(1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(1, keepdims=True)

def fit_temperature(logits, y, bounds=(0.1, 10.0)):
    idx = np.arange(len(y))
    def nll(T):
        p = softmax(logits / T)
        return float(-np.log(p[idx, y] + 1e-12).mean())
    return float(minimize_scalar(nll, bounds=bounds, method='bounded').x)

def stratified_sample(labels, fraction, seed):
    """Same routine used for the training subset, applied to either split."""
    np.random.seed(seed)
    idx_all = np.arange(len(labels))
    picked, forced = [], []
    for c in np.unique(labels):
        ci = idx_all[labels == c]
        exact = len(ci) * fraction
        n = max(1, int(exact))
        if exact < 1:
            forced.append(int(c))
        picked.extend(np.random.choice(ci, size=n, replace=False))
    return np.array(sorted(picked)), forced

def degeneracy(pred, k):
    cnt = np.bincount(pred, minlength=k)
    return float(cnt.max() / cnt.sum())

def load_emb(mkey, dkey, split):
    p = f'{EMB_DIR}/{mkey}/{dkey}'
    return (np.load(f'{p}/{split}_features.npy', mmap_mode='r'),
            np.load(f'{p}/{split}_labels.npy').astype(int))

def proba_chunked(clf, X, chunk=20000):
    return np.vstack([clf.predict_proba(np.asarray(X[i:i+chunk], dtype=np.float32))
                      for i in range(0, X.shape[0], chunk)])

def logits_chunked(clf, X, chunk=20000):
    out = []
    for i in range(0, X.shape[0], chunk):
        d = clf.decision_function(np.asarray(X[i:i+chunk], dtype=np.float32))
        if d.ndim == 1:
            d = d.reshape(-1, 1); d = np.hstack([-d, d])
        out.append(d)
    return np.vstack(out)

print('ready')

ready


## 2. Run under a proportional budget

The only change from the corrected protocol: the validation split is
subsampled by the same fraction and seed as the training split. Every row
records `n_train` and `n_val` so the realised budget is visible rather than
implied.

In [ ]:
def run_cell(model, dataset):
    mk, dk = M2K[model], D2K[dataset]
    Xtr, ytr = load_emb(mk, dk, 'train')
    Xte, yte = load_emb(mk, dk, 'test')
    try:
        Xva, yva = load_emb(mk, dk, 'val')
    except FileNotFoundError:
        Xva, yva = None, None

    n_classes = len(np.unique(ytr))
    rows = []

    for frac in FRACTIONS:
        for seed in SEEDS:
            idx, forced = stratified_sample(ytr, frac, seed)
            clf = LogisticRegression(max_iter=MAX_ITER, C=C_DEFAULT,
                                     random_state=seed)
            clf.fit(np.asarray(Xtr[idx], dtype=np.float32), ytr[idx])

            proba = proba_chunked(clf, Xte)
            pred  = proba.argmax(1)
            try:
                auroc = (roc_auc_score(yte, proba[:, 1]) if n_classes == 2
                         else roc_auc_score(yte, proba, multi_class='ovr',
                                            average='macro'))
            except Exception:
                auroc = np.nan

            # --- the change: validation subsampled by the same fraction ---
            if Xva is not None:
                vidx, vforced = stratified_sample(yva, frac, seed)
                Lva = logits_chunked(clf, np.asarray(Xva[vidx], dtype=np.float32))
                T = fit_temperature(Lva, yva[vidx])
                sp = softmax(logits_chunked(clf, Xte) / T)
                ece_s_fix, ece_s_ada = ece_fixed(sp, yte), ece_adaptive(sp, yte)
                n_val, n_vforced = len(vidx), len(vforced)
                del Lva, sp
            else:
                T = ece_s_fix = ece_s_ada = np.nan
                n_val = n_vforced = 0

            rows.append(dict(
                budget='proportional', model=model, dataset=dataset,
                fraction=frac, seed=seed, n_train=len(idx), n_val=n_val,
                n_labels_total=len(idx) + n_val,
                n_classes=n_classes, auroc=auroc,
                f1_macro=f1_score(yte, pred, average='macro', zero_division=0),
                brier=(brier_score_loss(yte, proba[:, 1])
                       if n_classes == 2 else np.nan),
                ece_fixed=ece_fixed(proba, yte),
                ece_adaptive=ece_adaptive(proba, yte),
                temperature=T,
                ece_scaled_fixed=ece_s_fix,
                ece_scaled_adaptive=ece_s_ada,
                degeneracy_share=degeneracy(pred, n_classes),
                forced_classes=len(forced), forced_val_classes=n_vforced))
            del proba, clf
            gc.collect()

    del Xtr, Xte, Xva
    gc.collect()
    df = pd.DataFrame(rows)
    df['degenerate'] = df['degeneracy_share'] > 0.99
    return df


t0 = time.time()
for dataset in ORDER:
    for model in MODELS:
        out = f'{CKPT}/{M2K[model]}__{D2K[dataset]}.csv'
        if os.path.exists(out):
            print(f'  skip: {model} x {dataset}')
            continue
        try:
            df = run_cell(model, dataset)
            df.to_csv(out, index=False)
            lo = df[df.fraction == 0.01].iloc[0]
            s = df.groupby('fraction')['temperature'].mean()
            print(f'{model:>12} x {dataset:<14} '
                  f'1%: train={int(lo.n_train)} val={int(lo.n_val)}  '
                  f'T {s.loc[0.01]:.2f} -> {s.loc[1.00]:.2f}  '
                  f'({time.time()-t0:.0f}s)')
        except Exception as e:
            print(f'{model:>12} x {dataset:<14} FAILED: {type(e).__name__}: {e}')
        gc.collect()

parts = sorted(glob.glob(f'{CKPT}/*.csv'))
df_pv = pd.concat([pd.read_csv(p) for p in parts], ignore_index=True)
df_pv.to_csv(f'{OUT_DIR}/indomain_propval.csv', index=False)
print(f'\n{len(parts)}/48 cells, {len(df_pv)} runs -> indomain_propval.csv')

        CLIP x LungHist700    1%: train=7 val=7  T 1.87 -> 0.36  (9s)
        PLIP x LungHist700    1%: train=7 val=7  T 0.11 -> 0.41  (17s)
       CONCH x LungHist700    1%: train=7 val=7  T 0.10 -> 0.48  (26s)
    VIRCHOW2 x LungHist700    1%: train=7 val=7  T 0.14 -> 0.46  (45s)
         UNI x LungHist700    1%: train=7 val=7  T 0.17 -> 0.42  (57s)
    GigaPath x LungHist700    1%: train=7 val=7  T 0.35 -> 0.39  (69s)
 H-Optimus-0 x LungHist700    1%: train=7 val=7  T 0.18 -> 0.37  (81s)
    MIDNIGHT x LungHist700    1%: train=7 val=7  T 0.24 -> 0.51  (93s)
        CLIP x MHIST          1%: train=18 val=3  T 2.54 -> 1.51  (103s)
        PLIP x MHIST          1%: train=18 val=3  T 2.34 -> 1.44  (114s)
       CONCH x MHIST          1%: train=18 val=3  T 2.29 -> 1.76  (122s)
    VIRCHOW2 x MHIST          1%: train=18 val=3  T 2.50 -> 1.21  (136s)
         UNI x MHIST          1%: train=18 val=3  T 2.18 -> 1.37  (145s)
    GigaPath x MHIST          1%: train=18 val=3  T 2.28 -> 1.36  (1

## 3. What the realised label budget actually was

The table the reviewer's comment implies should exist. `n_val` under the
original protocol is the full validation split at every fraction.

In [ ]:
orig = pd.read_csv(f'{OUT_DIR}/indomain_all_v2.csv')

val_sizes = (df_pv[df_pv.fraction == 1.00]
             .groupby('dataset')['n_val'].first())

rows = []
for ds in DATASETS:
    if ds not in val_sizes.index:
        continue
    full_val = int(val_sizes[ds])
    for f in FRACTIONS:
        n_tr = int(df_pv[(df_pv.dataset == ds) & (df_pv.fraction == f)]
                   ['n_train'].mean())
        n_va = int(df_pv[(df_pv.dataset == ds) & (df_pv.fraction == f)]
                   ['n_val'].mean())
        rows.append(dict(dataset=ds, fraction=f, n_train=n_tr,
                         val_original=full_val, val_proportional=n_va,
                         total_original=n_tr + full_val,
                         total_proportional=n_tr + n_va,
                         val_share_original=full_val / (n_tr + full_val)))
budget = pd.DataFrame(rows)
budget.to_csv(f'{OUT_DIR}/label_budget_accounting.csv', index=False)

print('Realised label budget, original protocol against proportional\n')
for ds in DATASETS:
    sub = budget[budget.dataset == ds]
    if sub.empty:
        continue
    print(f'--- {ds} ---')
    print(sub[['fraction','n_train','val_original','val_proportional',
               'total_original','total_proportional','val_share_original']]
          .rename(columns={'val_share_original':'val_frac_of_total'})
          .round(3).to_string(index=False))
    print()

print('The val_frac_of_total column at 1% is the quantity the reviewer is '
      'pointing at:\nthe share of the labelled data that came from the '
      'validation split.')

Realised label budget, original protocol against proportional

--- PCam ---
 fraction  n_train  val_original  val_proportional  total_original  total_proportional  val_frac_of_total
     0.01     2620         32768               326           35388                2946              0.926
     0.05    13106         32768              1637           45874               14743              0.714
     0.10    26214         32768              3275           58982               29489              0.556
     0.25    65536         32768              8191           98304               73727              0.333
     0.50   131072         32768             16383          163840              147455              0.200
     1.00   262144         32768             32768          294912              294912              0.111

--- BRACS ---
 fraction  n_train  val_original  val_proportional  total_original  total_proportional  val_frac_of_total
     0.01       33           312                 7           

## 4. Consistency check, then the comparison that matters

AUROC and raw ECE do not touch the validation split, so they must reproduce
the original run exactly. If they do not, something else changed and the
comparison below is not interpretable.

In [ ]:
key = ['model','dataset','fraction','seed']
a = orig.set_index(key).sort_index()
b = df_pv.set_index(key).sort_index()
common = a.index.intersection(b.index)
a, b = a.loc[common], b.loc[common]

print('=== consistency check on columns that cannot change ===')
for col in ['auroc', 'ece_fixed', 'f1_macro']:
    d = (a[col] - b[col]).abs()
    print(f'  {col:<12} max |diff| = {d.max():.2e}  '
          f'{"IDENTICAL" if d.max() < 1e-9 else "DIFFERS - investigate"}')

print('\n=== what the proportional budget changes ===')
for col in ['temperature', 'ece_scaled_fixed', 'ece_scaled_adaptive']:
    d = (b[col] - a[col])
    t = d.groupby('fraction').agg(['mean','std','min','max'])
    print(f'\n--- {col}: proportional minus original ---')
    print(t.round(4).to_string())

print('\n=== scaled ECE by fraction, both protocols ===')
cmp = pd.DataFrame({
    'original':     a.groupby('fraction')['ece_scaled_fixed'].mean(),
    'proportional': b.groupby('fraction')['ece_scaled_fixed'].mean()})
cmp['delta'] = cmp['proportional'] - cmp['original']
print(cmp.round(4).to_string())

print('\n=== temperature by fraction ===')
tt = pd.DataFrame({
    'original':     a.groupby('fraction')['temperature'].mean(),
    'proportional': b.groupby('fraction')['temperature'].mean()})
tt['delta'] = tt['proportional'] - tt['original']
print(tt.round(3).to_string())

print('\n=== how often does the temperature hit a search bound? ===')
for name, df in [('original', a), ('proportional', b)]:
    hit = ((df['temperature'] <= 0.101) | (df['temperature'] >= 9.999))
    print(f'  {name:<14} {hit.groupby(df.index.get_level_values("fraction")).mean().round(3).to_dict()}')

=== consistency check on columns that cannot change ===
  auroc        max |diff| = 6.68e-06  DIFFERS - investigate
  ece_fixed    max |diff| = 1.48e-04  DIFFERS - investigate
  f1_macro     max |diff| = 2.38e-04  DIFFERS - investigate

=== what the proportional budget changes ===

--- temperature: proportional minus original ---
            mean     std     min     max
fraction                                
0.01      0.1042  0.8931 -0.7758  8.9387
0.05      0.1814  1.1248 -0.5459  9.2469
0.10      0.0025  0.0935 -0.3634  0.4012
0.25     -0.0257  0.0711 -0.3218  0.1118
0.50     -0.0219  0.0633 -0.3354  0.0974
1.00     -0.0000  0.0000 -0.0001  0.0000

--- ece_scaled_fixed: proportional minus original ---
            mean     std     min     max
fraction                                
0.01      0.0092  0.0249 -0.0784  0.1197
0.05     -0.0047  0.0298 -0.1749  0.0512
0.10     -0.0011  0.0188 -0.1064  0.0445
0.25      0.0030  0.0130 -0.0345  0.0547
0.50      0.0007  0.0094 -0.0533  0.056

## 5. Does the calibration story change?

Two claims depend on the validation split: recoverability (raw minus scaled
ECE) and any ranking by scaled ECE. Both are recomputed here.

In [ ]:
from scipy.stats import spearmanr, kendalltau

print('=== Recoverability: raw ECE minus scaled ECE, by fraction ===')
rec = pd.DataFrame({
    'original':     (a['ece_fixed'] - a['ece_scaled_fixed']).groupby('fraction').mean(),
    'proportional': (b['ece_fixed'] - b['ece_scaled_fixed']).groupby('fraction').mean()})
rec['delta'] = rec['proportional'] - rec['original']
print(rec.round(4).to_string())
print('\nPositive values mean temperature scaling helps. If the proportional '
      'column is\nsmaller at low fractions, part of the reported recoverability '
      'came from validation\nlabels the 1% condition should not have had.')

print('\n=== Does the calibration ranking of models survive? ===')
print('Kendall tau between the two protocols, per (dataset, fraction)\n')
rows = []
for ds in DATASETS:
    for f in FRACTIONS:
        x = (a.reset_index().query('dataset == @ds and fraction == @f')
             .groupby('model')['ece_scaled_fixed'].mean())
        y = (b.reset_index().query('dataset == @ds and fraction == @f')
             .groupby('model')['ece_scaled_fixed'].mean())
        k = x.index.intersection(y.index)
        if len(k) < 3:
            continue
        rows.append(dict(dataset=ds, fraction=f,
                         tau=kendalltau(x[k], y[k]).correlation))
tt2 = pd.DataFrame(rows)
print(tt2.pivot_table(index='dataset', columns='fraction', values='tau')
      .round(3).to_string())
print(f"\n  mean tau {tt2['tau'].mean():.3f}, "
      f"sign inversions {(tt2['tau'] < 0).sum()} of {len(tt2)}")

print('\n=== Decoupling correlation under the proportional budget ===')
def rho_at(df, fraction, ece_col='ece_scaled_fixed'):
    out = []
    d = df.reset_index()
    for ds in DATASETS:
        s = (d[(d.dataset == ds) & (d.fraction == fraction)]
             .groupby('model')[['auroc', ece_col]].mean().dropna())
        if len(s) < 3:
            continue
        out += list(zip(s['auroc'].rank(ascending=False).values,
                        s[ece_col].rank(ascending=True).values))
    if len(out) < 4:
        return np.nan
    x, y = zip(*out)
    return spearmanr(x, y).correlation

print(f"{'frac':>6} {'original':>10} {'proportional':>14}")
for f in FRACTIONS:
    print(f'{f:>6.2f} {rho_at(a, f):>10.3f} {rho_at(b, f):>14.3f}')

b.reset_index().to_csv(f'{OUT_DIR}/indomain_propval.csv', index=False)
print('\nSaved -> indomain_propval.csv, label_budget_accounting.csv')

=== Recoverability: raw ECE minus scaled ECE, by fraction ===
          original  proportional   delta
fraction                                
0.01        0.1251        0.1159 -0.0092
0.05        0.0547        0.0594  0.0047
0.10        0.0491        0.0503  0.0011
0.25        0.0435        0.0405 -0.0030
0.50        0.0383        0.0375 -0.0007
1.00        0.0330        0.0329 -0.0000

Positive values mean temperature scaling helps. If the proportional column is
smaller at low fractions, part of the reported recoverability came from validation
labels the 1% condition should not have had.

=== Does the calibration ranking of models survive? ===
Kendall tau between the two protocols, per (dataset, fraction)

fraction        0.01   0.05   0.10   0.25   0.50  1.00
dataset                                               
BRACS          0.500  0.643  0.214  0.714  0.429   1.0
CRC            0.357  0.643  0.857  0.786  0.929   1.0
LungHist700    0.571  0.143  0.214  0.500  0.786   1.0
MHIST  

## 6. Reading the result

**If the scaled-ECE numbers barely move**, the reviewer's point is real as an
accounting matter and immaterial as an empirical one, and that is worth saying
plainly with the table: the 1% condition consumed more labels than its name
implies, and correcting the accounting changes the numbers by *x*.

**If they move materially**, the label-efficiency framing needs the
proportional budget as its primary protocol, and the original numbers become
the ablation rather than the headline.

Either way the accounting table in section 3 should go into the manuscript.
The reviewer is right that "1%" was doing work the protocol did not support,
and a reader cannot check that without seeing `n_val` alongside `n_train`.

In [ ]:
!find . -iname "*propval.ipynb" -print -quit

./drive/MyDrive/Colab Notebooks/PRISM_rebuttal_propval.ipynb


In [ ]:
d = (b['auroc'] - a['auroc'])
print(d.mean())
print(d.groupby('fraction').mean().round(8).to_dict())
print((d.abs() > 1e-9).mean().round(3))